## Statistical data analysis. Exam 

**(!) <span style="color:blue"> Add your name to the file name instead of [Name]</span>**</br>
Example: Stat_analysis_25-26_exam_Ivanov Ivan.ipynb

Submit your notebook to Smart LMS https://edu.hse.ru/course/view.php?id=268179 (access at the end of assessment only)

Unlocked web sites to visit:</br>
* https://docs.scipy.org/doc/scipy/tutorial/index.html
* https://docs.python.org/3/library/random.html
* https://docs.python.org/3/library/
* https://numpy.org/
* https://pandas.pydata.org/docs/user_guide/index.html
* https://matplotlib.org/
* https://plotly.com/python/

In [ ]:
import numpy as np
import random
import plotly.graph_objects as go

**Question 1 [11 pts]**. 

In [ ]:
from scipy.stats import rv_continuous
from scipy.stats import norm

https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_continuous.html

Consider the kernel density estimator $\widehat{f}_h(x)$ (see Problem 1b) with $n=2,\,x_1=0.5,\,x_2=0.6:$
$$
\widehat f_h(x) = \frac{1}{2h}\left(\varphi\left(\frac{x-0.5}{h}\right)+\varphi\left(\frac{x-0.6}{h}\right)\right),
$$
where $\varphi(x)$ is a pdf of standard normal random variable.

1) Derive a class from rv_continuous (call it *kd_estimator*) which represents distribution above. Bandwidth $h$ will be a member variable of this class.</br>
2) Create an instance of this class with $h=0.5$ and generate a sample $X_1,\ldots, X_k$ of size $k=15$.

In [ ]:
class kd_estimator(rv_continuous):
    def __init__(self, bandwidth):
        super().__init__()
        self.h = bandwidth
    
    def _pdf(self, x):
        return (norm.pdf((x-0.5)/self.h) + norm.pdf((x-0.6)/self.h))/(2*self.h)

In [ ]:
h = 0.5
distr = kd_estimator(h)

In [ ]:
sample = distr.rvs(size = 15)

3) Output difference $|\overline X - \mathsf E(X)|,$ where $\mathsf E(X)$ is computed using *kd_estimator* class functionality. 

In [ ]:
mu = distr.mean()
x_bar = np.mean(sample)
print(abs(mu-x_bar))

**Question 2 [34 pts]** 

**2.1) [13 pts]** Geometric Brownian motion $(S_t)_{t\ge 0}$ for pricing purposes satisfies the following stochastic differential equation (SDE):
$$
dS_t = rS_tdt+\sigma S_tdW_t
$$
Write a function *spot_simulation(s0, r, sigma, m, T, n_paths)* which sumulates   $n\_paths$ of $(S_t)_{0\le t\le T}$ SDE up to time $T$, which is divided by points with step $h=\frac1m$.

In [ ]:
def spot_simulation(s0, r, sigma, m, T, n_paths):
    h = 1/m
    paths = []
    for i in range(n_paths):
        path = [s0]
        ksi = norm.rvs(0, h ** 0.5, m * T)

        for j in range(1, T*m + 1):
            val = path[-1] + r * path[-1] * h + sigma * ksi[j - 1] * path[-1]
            path.append(val)
        paths.append(np.array(path))
    
    return np.array(paths)

Let $S_0=100, r = 0.15,\sigma = 0.6, m = 100, T=2$. Simulate 3 paths and output them at one graph. 

In [ ]:
paths = spot_simulation(100, 0.15, 0.6, 100, 2, 3)
t = np.linspace(0, 2, 201)

plot = go.Figure()
for path in paths:
    graph = go.Scatter(x = t, y = path)
    plot.add_trace(graph)
plot.show()

**Pricing of an instrument**. Instrument class as well as generic pricer class are given:

In [ ]:
class instrument:
    def __init__ (self, strike, maturity):
        self.K = strike
        self.T = maturity
        
    def payoff(self, S):
        if S > self.K:
            return 1
        else:
            return 0

In [ ]:
class pricer:
    def __init__(self):
        pass
        
    def pv(self, instr):
        pass

**2.2) [15 pts]** The task is to price the instrument from Problem 3 via Monte-Carlo for instrument with strike $K=90$ and maturity $T=2$ years. Since the payments can happen at $T/2$ and at $T$ we need to use the whole path simulation i.e. using *spot_simulation()* call.

Write *monte_carlo_pricer* class which derives from pricer and implements *pv()* function. You may use the variables below for *spot_simulation()* call.

In [ ]:
S0 = 100
r = 0.15
sigma = 0.6
m = 100

In [ ]:
class monte_carlo_pricer(pricer):
    def __init__(self, N_paths):
        super().__init__()
        self.N = N_paths  
        
    def pv(self, instr):        
        paths = spot_simulation(S0, r, sigma, m, instr.T, self.N)
        middles = paths[:, m*T//2]
        disc_payoff_mid = [np.exp(-r*instr.T/2)*instr.payoff(mid) for mid in middles]
        
        lasts = paths[:, m*T]
        disc_payoff_last = [np.exp(-r*instr.T)*instr.payoff(last) for last in lasts]
        
        return np.mean(disc_payoff_mid) + np.mean(disc_payoff_last)

Compute and output $PV$ of an instrument using 10 000 simulations.

In [ ]:
K = 90
T = 2
N = 10_000
instr = instrument(K, T)
mc_pricer = monte_carlo_pricer(N)
mc_pv = mc_pricer.pv(instr)
mc_pv

**2.3) [6 pts]** Write *analytic* class which derives from pricer and implements *pv()* function which you found in Problem 3b). During implementation you may use global variables $S_0, r, sigma$ defined above, the rest should be taken from instrument.

In [ ]:
class analytic_pricer(pricer):
    def __init__(self):
        super().__init__()
    
    def pv(self, instr):
        d_half = (np.log(S0/instr.K) + (r-sigma**2/2)*instr.T/2)/(sigma*np.sqrt(T/2))
        d = (np.log(S0/instr.K) + (r-sigma**2/2)*instr.T)/(sigma*np.sqrt(T))
        
        return np.exp(-r*instr.T/2)*norm.cdf(d_half) + np.exp(-r*instr.T)*norm.cdf(d)

In [ ]:
an_pricer = analytic_pricer()
an_pv = an_pricer.pv(instr)
an_pv

**Question 3 [10 pts]** 

Consider differential equation $y^{\prime} - 2y=e^{2x},\,y(0)=0,\,0\le x\le 2.$ The exact solution of this equation is $y(x)=xe^{2x}.$ This is a typical example of the phenomenon known as *resonance*.

In order to solve differential equation we consider the following finite difference scheme:
$$
\frac{y_i-y_{i-1}}{h} - 2y_{i-1}=e^{2x_i},\, x_i=ih,\, i=1,\ldots,2n,\,h=1/n,
$$
$$
y_0=0.
$$
Implement finite difference scheme. Output 4 scatterplots on the same plot: exact solution $y=y(x)$ and 3 solutions $(y_0,\ldots, y_{2n})$ of finite difference schemes for $n=10,\,20$ and $50$.

In [ ]:
ns = [10, 20, 50]
T = 2
solution = []
for n in ns:
    h = 1/n
    yn = [0]
    xn = [0]
    for i in range(1, T*n+1):
        xn.append(i*h)
        yn.append(yn[-1]*(1+2*h) + h*np.exp(2*i*h))
    solution.append([xn, yn])

In [ ]:
plot = go.Figure()

x_ = np.linspace(0,2,1001)
y_ = x_*np.exp(2*x_)
graph = go.Scatter(x = x_, y = y_, name = 'Exact solution')
plot.add_trace(graph)

for i in range(0, len(ns)):
    graph_2 = go.Scatter(x = solution[i][0], y = solution[i][1], name = f'Solution n={ns[i]}')
    plot.add_trace(graph_2)

plot.show()

Helper functon: Brownian motion simulation

In [ ]:
# Call, but DO NOT MODIFY THIS FUNCTION
def bm_simulations(n_paths, granularity, max_time):
    n_points = granularity * max_time
    
    paths = []
    for _ in range(n_paths):
        xs = [0] + random.choices([-1, 1], weights = [0.5, 0.5], k = n_points)
        path = np.cumsum(xs)/np.sqrt(granularity)
        paths.append(path)
    
    return np.array(paths)